RAG Pipeline - VectorDB to LLm Output Generation

In [15]:
import os ,sys
from dotenv import load_dotenv
load_dotenv()
print(os.getenv("GROQ_API-KEY"))
sys.path.insert(0, os.path.abspath(".."))


None


In [16]:
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage,SystemMessage

In [17]:
class GroqLLM:
    def __init__(self,model_name:str = "llama-3.3-70b-versatile",api_key:str=None):
        """
        Initialize groq LLM
        Args:
            model_name:Groq model name (qwen2=-72b-instruct,llama3-70b-8192,etc.)
        """

        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required , Set GROQ_API_KEY enivironment variable or pass api_key_key parameter . ")
        
        self.llm=ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model:{self.model_name}")
    
    def generate_response(self, query:str, context : str , max_length:int=500) -> str:
        """
        Generate response using retrived context

        Args:
            query:User question
            context: Retrieved document context
            max_length : maximum response length
        Returns:
            Generated response string 
        """

        #Create prompt template
        prompt_template=PromptTemplate(
            input_variables=["context","question"],
            template="""You are a helpful AI assistant .Use the following context to answer the question accurately and concisely.
            
            Context:{context}
            Question:{question}
            Answer: Provide a clear and informative answer based on the context above . If the context doesn't contain enough information to answer the question , say so."""
        )
        #Format the prompt 
        formatted_prompt = prompt_template.format(context=context, question=query)

        try :
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error generating response :{str(e)}"
    
    def generate_response_simple(self,query: str, context :str) -> str:
        """
        Simple response generation without complex prompting 
        
        Args:
            query : User question
            context:Retrieved context
        
        Respnse:
            Generated response 
        """
        simple_prompt = f"""Based on this context :{context} Question:{query} Answer: """

        try :
            messages=[HumanMessage(content=simple_prompt)]
            response=self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error :{str(e)}"
       

In [18]:
# Initialize Groq LLm (you'll need to set GROQ_API_KEY enviornment variable )
try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully !!")
except ValueError as e:
    print(f"Warning:{e}")
    print("Please set your GROQ_API_KEY enviorment variable to use the LLM.")
    groq_llm = None
    

Initialized Groq LLM with model:llama-3.3-70b-versatile
Groq LLM initialized successfully !!


In [19]:
from rag_utils import RAGRetriever, VectorStore, embedding_manager

rag_retriever=RAGRetriever(VectorStore,embedding_manager)
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query:'Unified Multi-task Learning Framework
Top K:5,Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 45.89it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_301e0455_225',
  'content': 'which provides additional information by consulting nearby\nobjects and surroundings (GBD-Net and multi-path).\n• Due to the existence of a large number of nonstandard\nsmall objects, the results on this dataset are much worse\nthan those of VOC 2007/2012. With the introduction of\nother powerful frameworks (e.g. ResNeXt [123]) and useful\nstrategies (e.g. multi-task learning [67], [124]), the perfor-\nmance can be improved.\n• The success of DSOD in training from scratch stresses the',
  'metadata': {'author': '',
   'modDate': 'D:20190417004522Z',
   'moddate': '2019-04-17T00:45:22+00:00',
   'creationdate': '2019-04-17T00:45:22+00:00',
   'source': '..\\data\\pdf\\1807.05511v2.pdf',
   'page': 9,
   'content_length': 481,
   'creationDate': 'D:20190417004522Z',
   'doc_index': 225,
   'title': '',
   'file_path': '..\\data\\pdf\\1807.05511v2.pdf',
   'keywords': '',
   'producer': 'pdfTeX-1.40.17',
   'total_pages': 21,
   'trapped': '',
   

Integration Vectordb Context pipeline With LLM output 

In [20]:
##Simple Rag pipeline with Groq LLM 
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

groq_api_key= os.getenv("GROQ_API_KEY")

llm=ChatGroq(groq_api_key=groq_api_key,model_name="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)

def rag_simple(query,retriever,llm,top_k=3):
    ##retriever the context
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results ])if results else ""

    if not context:
        return "No relevent context found to answer the question."
    
    ##generate the answer using Groq LLM
    prompt=f"""Use the following context to answer the questio concisely .
        Context:{context}
        question;{query}
        Answer:"""
    
    response=llm.invoke([prompt.format(context=context,query = query)])
    return response.content


In [21]:
answer=rag_simple("What is attention mechanism ? ",rag_retriever,llm)
print(answer)

Retrieving documents for query:'What is attention mechanism ? 
Top K:3,Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 56.26it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


The attention mechanism is a process where individual attention heads learn to perform different tasks, exhibiting behavior related to the syntactic and semantic structure of sentences, and following long-distance dependencies in the encoder self-attention.
